# MLP Style NN for name generation

In [1]:
import torch
import matplotlib.pyplot as plt
import torch.nn.functional as F
%matplotlib inline

In [2]:
words = open("names.txt", "r").read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [3]:
len(words)

32033

In [4]:
chars = sorted(list(set("".join(words))))
stoi = {s:i+1 for i, s in enumerate(chars)}
stoi["."]=0
itos = {i:s for s, i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [5]:
block_size = 3
X, Y = [], []
for word in words[:5]:
    context = [0]*block_size
    print(word)
    for ch in word+".":
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        print("".join(itos[i] for i in context), "--->", itos[ix])
        context = context[1:]+[ix]
X = torch.tensor(X)
Y = torch.tensor(Y)

emma
... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .
olivia
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .
ava
... ---> a
..a ---> v
.av ---> a
ava ---> .
isabella
... ---> i
..i ---> s
.is ---> a
isa ---> b
sab ---> e
abe ---> l
bel ---> l
ell ---> a
lla ---> .
sophia
... ---> s
..s ---> o
.so ---> p
sop ---> h
oph ---> i
phi ---> a
hia ---> .


In [6]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([32, 3]), torch.int64, torch.Size([32]), torch.int64)

In [7]:
C = torch.randn(27,2)

In [8]:
C

tensor([[-0.8134,  0.2953],
        [ 0.4230,  0.8785],
        [-1.1808, -2.2209],
        [-1.2329,  0.8107],
        [-1.1339, -0.2806],
        [ 1.4081,  0.3829],
        [ 0.3997,  1.3168],
        [ 1.4473, -0.5466],
        [-0.3348,  0.2295],
        [ 0.6789,  1.4022],
        [-0.6737, -0.6793],
        [ 0.9943, -0.1777],
        [ 0.9407,  0.0544],
        [ 0.0129,  0.3377],
        [ 0.0286,  0.2673],
        [ 0.6997, -1.8696],
        [ 0.9983,  0.4508],
        [ 0.8424,  1.3456],
        [ 1.3232,  0.2155],
        [ 1.1363,  0.1919],
        [ 0.6871, -2.6718],
        [ 1.2221,  1.3600],
        [-0.8007, -1.3607],
        [-1.1460,  2.2916],
        [-0.2183, -0.5774],
        [ 1.2747,  0.0455],
        [ 0.3627, -2.2334]])

In [9]:
F.one_hot(torch.tensor(5), num_classes=27).float() @ C

tensor([1.4081, 0.3829])

In [10]:
emb = C[X]
emb.shape

torch.Size([32, 3, 2])

In [11]:
W1 = torch.randn(6, 100)
b1 = torch.randn(100)

In [12]:
h=torch.tanh(emb.view(-1, 6)@W1+b1)
h

tensor([[-0.9675, -0.9998,  0.1868,  ..., -0.9876,  0.7823, -0.2835],
        [-0.9707, -0.5674, -0.9880,  ..., -0.9956, -0.8846,  0.2014],
        [ 0.8112,  0.7238,  0.9414,  ..., -0.4590,  0.9988, -0.9954],
        ...,
        [ 0.2246,  0.6318,  0.3616,  ..., -0.3923,  0.9999, -0.9835],
        [-0.9507, -0.9937, -0.9920,  ...,  0.2986, -0.8281, -0.8261],
        [-0.9070,  0.8007, -0.7309,  ..., -0.7295,  0.9952, -0.9947]])

In [13]:
h.shape

torch.Size([32, 100])

In [14]:
W2 = torch.randn(100, 27)
b2 = torch.randn(27)

In [15]:
logits = h @ W2 + b2

In [16]:
logits.shape

torch.Size([32, 27])

In [17]:
counts = logits.exp()
counts

tensor([[2.0802e+00, 2.9535e-08, 5.1168e-05, 1.1858e+06, 1.1560e+03, 2.5359e-02,
         1.5790e-06, 2.6409e-07, 5.3070e-02, 3.6357e-05, 1.8230e+05, 5.1835e+07,
         9.4091e+05, 1.4521e+00, 1.1964e+06, 1.6928e+02, 2.1915e+03, 6.4774e-02,
         7.4209e+01, 2.0411e+01, 1.4369e+03, 4.3070e+02, 4.5609e+00, 2.1162e+01,
         3.9082e+00, 2.5529e-04, 2.4882e+04],
        [1.0111e+00, 1.2216e-06, 3.2534e+05, 9.7652e+01, 2.4529e-04, 2.0908e+01,
         5.8417e+01, 1.4502e-06, 2.6652e-03, 3.6360e-07, 4.0438e+02, 9.8554e+04,
         9.7380e+01, 6.4265e-03, 3.4961e-03, 4.2407e-01, 2.1857e-01, 2.2074e+07,
         6.2188e+00, 2.1311e-02, 1.9169e-01, 5.4109e+02, 4.2752e-01, 1.0501e+00,
         2.7459e-05, 3.4060e+00, 4.6094e+02],
        [1.2710e-02, 1.1850e-05, 1.0310e+05, 2.5677e-05, 9.7388e+00, 2.3432e+01,
         1.5514e-05, 2.5124e-05, 2.1914e-04, 3.5388e+00, 2.9453e+04, 1.3351e+09,
         1.2387e-02, 3.1102e-02, 3.6691e+00, 7.7428e+00, 3.8415e+03, 1.0357e-02,
         2.9690e+

In [18]:
prob = counts / counts.sum(1, keepdims=True)

In [19]:
prob.shape

torch.Size([32, 27])

In [20]:
Y

tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0,  9, 19,
         1,  2,  5, 12, 12,  1,  0, 19, 15, 16,  8,  9,  1,  0])

In [21]:
prob[torch.arange(32), Y].log().mean()

tensor(-21.2434)

In [22]:
##### Makeing abovemore respectable

In [23]:
g=torch.Generator().manual_seed(2147483647)
C=torch.randn((27,10), generator=g)
W1=torch.randn((30, 200), generator=g)
b1 = torch.randn(200, generator=g)
W2=torch.randn((200, 27), generator=g)
b2=torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

In [24]:
sum(p.nelement() for p in parameters)

11897

In [25]:
for p in parameters:
    p.requires_grad = True

In [26]:
for _ in range(1000):
    emb = C[X]
    h=torch.tanh(emb.view(-1, 30) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Y)
    for p in parameters:
        p.grad = None

    loss.backward()
    for p in parameters:
        p.data += -0.1 * p.grad
    
print(loss.item())

0.2519923150539398


In [27]:
#### Sample from the trained model now
g=torch.Generator().manual_seed(2147483647+10)
for _ in range(20):
    out = []
    context = [0]*block_size
    while True:
        emb = C[torch.tensor([context])]
        h=torch.tanh(emb.view((1, -1)) @ W1 + b1)
        logits = h @ W2 + b2
        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1, generator=g).item()
        context = context[1:] + [ix]
        out.append(ix)
        if ix ==0:
            break
    print("".join(itos[i] for i in out))

olivia.
ava.
isabella.
sophia.
ava.
ava.
emma.
emma.
emma.
ava.
emma.
emma.
ava.
ava.
olivia.
sophia.
olivia.
ava.
sophia.
isabella.


In [28]:
X.shape

torch.Size([32, 3])